In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

dataset_path = "Image_dataset"

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=8,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=8,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=100,
    verbose=1,
    callbacks=callbacks
)

val_loss, val_acc = model.evaluate(val_generator, verbose=0)
print(f"acc: {val_acc:.2%}")

model.save("viral_detector_mobilenet_v2.keras")

Found 350 images belonging to 2 classes.
Found 86 images belonging to 2 classes.
Epoch 1/100


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 0.4764 - loss: 0.9606 - val_accuracy: 0.5116 - val_loss: 0.7713 - learning_rate: 1.0000e-04
Epoch 2/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.5208 - loss: 0.8101 - val_accuracy: 0.5698 - val_loss: 0.6744 - learning_rate: 1.0000e-04
Epoch 3/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 83ms/step - accuracy: 0.5384 - loss: 0.7555 - val_accuracy: 0.5581 - val_loss: 0.7227 - learning_rate: 1.0000e-04
Epoch 4/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.5802 - loss: 0.6856 - val_accuracy: 0.6047 - val_loss: 0.6700 - learning_rate: 1.0000e-04
Epoch 5/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.5837 - loss: 0.6989 - val_accuracy: 0.5465 - val_loss: 0.7100 - learning_rate: 1.0000e-04
Epoch 6/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.5674 - loss: 0.7325 - val_accuracy: 0.6512 - val_loss: 0.6454 - learning_rate: 1.0000e-04
Epoch 7/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/step - accuracy: 0.5898 - los

In [2]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "test.jpg"  

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = preprocess_input(img_array)
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array, verbose=0)
prob = prediction[0][0]

In [3]:
print(f"Trend (Viral) probability: {prob:.2%}")
print(f"Not Trend (Non-Viral) probability: {1-prob:.2%}")

if prob > 0.5:
    print("Final Result: ✅ Trend")
else:
    print("Final Result: ❌ Not Trend")

Trend (Viral) probability: 84.33%
Not Trend (Non-Viral) probability: 15.67%
Final Result: ✅ Trend
